# Ownership rules walkthrough

This notebook walks through the deductive core of the template on the **worked examples from the papers**:

* **F1**: Figure 1 of *Weaving Enterprise Knowledge Graphs* (Atzeni et al., EDBT 2020): company control, accumulated ownership, close links, family control.
* **F2**: the reconstructed Italian sample (their Figure 2), with a buy-back self-loop and a cross-holding cycle.
* **F3**: the Acme Bank case from *Rule-based AML in FIUs* (Bellomarini, Laurenza, Sallinger 2020): a loan applicant whose **family** controls the lending bank through a pyramid.
* **F4**: the slush-fund invoice rules from the same paper.

Each step quotes the paper's definition and shows the PyRel rule that implements it (in `model/`), then queries the result. Cells run against your Snowflake account (`raiconfig.yaml`); every query takes a few seconds.

In [ ]:
import pandas as pd
from config import Settings
from model import build_model
from model.explain import explain, render_text
from model.load import FrameSource
from relationalai.semantics import Float, Integer

FIX = "data/fixtures/"
def fixture(name, stages=None, **kw):
    return build_model(Settings(**kw), FrameSource.from_dir(FIX + name), name=f"bo_nb_{name}", stages=stages)

CORE = ["family", "holdings", "control", "ownership", "close_links"]

## 1. Company control (P1 Def 2.3; P3 Rules 4–6)

> *A company (or a person) x controls a company y if: (i) x directly owns more than 50% of y; or (ii) x controls a set of companies that jointly, and possibly together with x, own more than 50% of y.*

Condition (ii) is recursive **through a sum**, which PyRel does not support. `model/control.py` computes control as a **depth-bounded fixpoint**: layer *k* sums the shares held by what *x* controlled at layer *k−1*:

```python
tot = aggs.sum(z, w).per(x, y).where(prev(x, z), z.holds(y, w))    # P3 Rule 6 over layer k-1
m.where(tot > T).define(Lk(x, y))
```

In [ ]:
m1, o1 = fixture("f1_weaving_fig1", CORE)
a, b, d = o1.Entity.ref(), o1.Company.ref(), Integer.ref()
m1.where(a.control_depth(b, d), a.eid.in_(["P:P1", "P:P2"])).select(
    a.eid.alias("controller"), b.eid.alias("company"), d.alias("first_layer")).to_df().sort_values(["controller", "first_layer"])

P1 controls F at layer 3: 0.2 through D plus 0.4 through E, and E only becomes controlled at layer 2 (0.4 via D plus P1's own 0.2). The **control steps** are kept as evidence for explanations:

In [ ]:
s = o1.ControlStep.ref()
m1.where(s.controller.eid == "P:P1").select(s.target.eid.alias("company"), s.via.eid.alias("via"),
                                           s.share.alias("share"), s.depth.alias("layer")).to_df().sort_values(["layer", "company"])

## 2. Accumulated ownership and close links (P1 Def 2.5, 2.6)

> *Φ(x, y) = Σ over simple paths from x to y of the product of the shares along the path.* Two companies are **closely linked** (ECB collateral rule, T = 0.2) if one holds ≥ T of the other, or a third party holds ≥ T of both.

In [ ]:
x, y, v = o1.Entity.ref(), o1.Company.ref(), Float.ref()
m1.where(x.phi(y, v), x.eid == "P:P1").select(y.eid.alias("company"), v.alias("phi")).to_df().sort_values("phi", ascending=False)

In [ ]:
c1, c2 = o1.Company.ref(), o1.Company.ref()
m1.where(c1.close_link(c2), c1.eid < c2.eid).select(c1.eid.alias("a"), c2.eid.alias("b")).to_df()

## 3. Families (P3 Rules 1–3; P1 Def 2.8)

> *A family F controls y if a member controls y, or F's members and the companies F controls jointly own more than 50% of y.*

With P1 and P2 partners, the family controls **L** (0.2 via F, which P1 controls, plus 0.4 via H, which P2 controls), although neither controls it alone:

In [ ]:
f, p = o1.Family.ref(), o1.Person.ref()
fam = m1.where(p.family(f)).select(f.eid.alias("family"), p.eid.alias("member")).to_df()
ctl = m1.where(f.controls(y)).select(f.eid.alias("family"), y.eid.alias("company")).to_df()
fam, ctl.sort_values("company")

## 4. Cycles: why Φ has two methods (F2)

`PHI_METHOD="unrolled"` (default, scalable) multiplies shares hop by hop and drops walks that return to their source; a cycle among *intermediate* companies (C8 ↔ C9) still adds a little. `PHI_METHOD="paths"` sums exactly over simple paths with `model.path(...)` (Snowflake only).

In [ ]:
rows = []
for method in ["unrolled", "paths"]:
    m2, o2 = build_model(Settings(PHI_METHOD=method), FrameSource.from_dir(FIX + "f2_weaving_fig2"),
                         name=f"bo_nb_f2_{method}", stages=["holdings", "control", "ownership"])
    x, y, v = o2.Entity.ref(), o2.Company.ref(), Float.ref()
    df = m2.where(x.phi(y, v), x.eid.in_(["P:P3", "C:C4"]), y.eid.in_(["C:C9", "C:C7"])).select(
        x.eid.alias("src"), y.eid.alias("dst"), v.alias("phi")).to_df()
    rows.append(df.assign(method=method))
pd.concat(rows).pivot_table(index=["src", "dst"], columns="method", values="phi")

## 5. The Acme Bank case (P3 §3, Rule 10)

> *A person x who is issuing a loan request to a bank b of which he/she is the ultimate beneficial owner may intend to launder unclean money via the bank.*

x owns nothing of Acme Bank. Their **family** does: P2 holds 0.34 of My Bank, P1 (CEO of People Bank) reaches 0.23 more through Acme Trust, and My Bank controls Acme Bank with 0.52 through a pyramid.

In [ ]:
m3, o3 = fixture("f3_acme")
print(render_text(explain(m3, o3, "S1")))

## 6. Slush funds (P3 Eq. 1–2)

An invoice with no matching payment suggests false invoicing; a payment with no invoice is flagged too.

In [ ]:
m4, o4 = fixture("f4_slush")
x, t = o4.Entity.ref(), o4.Transfer.ref()
(m4.where(x.potential_slush_fund()).select(x.eid.alias("potential_slush_fund")).to_df(),
 m4.where(t.missing_invoice()).select(t.transfer_id.alias("paid_without_invoice")).to_df())

## Next

Run the whole pipeline on the synthetic sample (predicted family links, the GNN stages, scoring and the triage optimizer):

```bash
python beneficial_ownership_local.py
```